# Intervention session conclusion with Ollama

This notebook reads one synthetic mentor–student conversation from a Markdown file and asks a local Ollama model for an evidence-grounded JSON conclusion.

The default is `qwen3.5:latest`, selected for structured extraction and because it is available in the local Ollama installation. The notebook does not quantize or convert the model. Change `MODEL` below to try another Ollama tag.

The output distinguishes between:

- whether the **session** was useful (for example, it produced a concrete next step), and
- whether the **intervention** has demonstrated a helpful outcome (which should remain `unknown` if support was only proposed or requested).

In [1]:
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import os


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
MODEL = "qwen3.5:latest"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").rstrip("/")
DIALOGUE_PATH = PROJECT_ROOT / (
    "src/intervention/student_mentor_dialogues/concerns/"
    "02_attendance_transport_and_safety/transport_cost/session_0101.md"
)

print(f"Model: {MODEL}")
print(f"Dialogue: {DIALOGUE_PATH.relative_to(PROJECT_ROOT)}")

Model: qwen3.5:latest
Dialogue: src/intervention/student_mentor_dialogues/concerns/02_attendance_transport_and_safety/transport_cost/session_0101.md


## Check Ollama and load the transcript

Start Ollama before running this cell (normally `ollama serve`). Only the Markdown transcript is sent to the model; corpus metadata and manifest labels are not included.

In [2]:
def ollama_request(endpoint: str, payload: dict | None = None, timeout: int = 300) -> dict:
    url = f"{OLLAMA_BASE_URL}{endpoint}"
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(url, data=body, headers={"Content-Type": "application/json"})

    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {OLLAMA_BASE_URL}. Start it with `ollama serve`."
        ) from exc


available_models = {item["name"] for item in ollama_request("/api/tags").get("models", [])}
if MODEL not in available_models:
    raise RuntimeError(
        f"{MODEL!r} is not installed. Run `ollama pull {MODEL}` or choose one of: "
        f"{sorted(available_models)}"
    )

conversation = DIALOGUE_PATH.read_text(encoding="utf-8")
print(f"Loaded {len(conversation):,} characters. Ollama is ready.")

Loaded 9,199 characters. Ollama is ready.


## Define the output contract

The enums make uncertain or unobserved outcomes explicit. Evidence entries must be short verbatim excerpts from the conversation.

In [3]:
CONCLUSION_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "primary_concern": {
            "type": "string",
            "pattern": "^[a-z0-9]+(?:_[a-z0-9]+)*$",
            "description": "Main student problem or barrier, never the support/service offered",
        },
        "secondary_concerns": {
            "type": "array",
            "description": "Other explicit problems, excluding consequences of the primary concern",
            "items": {"type": "string", "pattern": "^[a-z0-9]+(?:_[a-z0-9]+)*$"},
        },
        "attendance_issue": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "reported_absences": {"type": ["integer", "null"], "minimum": 0},
                "cause": {"type": ["string", "null"]},
                "evidence_status": {
                    "type": "string",
                    "enum": ["student_reported", "mentor_reported", "record_verified", "mixed", "unknown"],
                },
            },
            "required": ["reported_absences", "cause", "evidence_status"],
        },
        "intervention": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "type": {
                    "type": "string",
                    "pattern": "^[a-z0-9]+(?:_[a-z0-9]+)*$",
                    "description": "Support, service, or intervention discussed",
                },
                "status": {
                    "type": "string",
                    "enum": [
                        "not_discussed", "information_only", "proposed", "inquiry_planned",
                        "assessment_requested", "referral_made", "active", "completed",
                        "declined", "unknown"
                    ],
                },
                "accepted_by_student": {"type": ["boolean", "null"]},
                "service_delivered": {"type": "boolean"},
            },
            "required": ["type", "status", "accepted_by_student", "service_delivered"],
        },
        "outcome": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "primary_barrier_resolved": {"type": "string", "enum": ["yes", "no", "partial", "unknown"]},
                "attendance_improved": {"type": "string", "enum": ["yes", "no", "partial", "unknown"]},
            },
            "required": ["primary_barrier_resolved", "attendance_improved"],
        },
        "helpfulness": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "session_helpful": {"type": "string", "enum": ["yes", "no", "mixed", "unknown"]},
                "intervention_helpful": {"type": "string", "enum": ["yes", "no", "mixed", "unknown"]},
                "rationale": {"type": "string"},
            },
            "required": ["session_helpful", "intervention_helpful", "rationale"],
        },
        "next_step": {
            "type": ["string", "null"],
            "pattern": "^[a-z0-9]+(?:_[a-z0-9]+)*$",
            "description": "Earliest concrete agreed action, in snake_case; null if none",
        },
        "evidence": {"type": "array", "items": {"type": "string"}, "maxItems": 5},
    },
    "required": [
        "primary_concern", "secondary_concerns", "attendance_issue", "intervention",
        "outcome", "helpfulness", "next_step", "evidence"
    ],
}

print(json.dumps(CONCLUSION_SCHEMA, indent=2))

{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "primary_concern": {
      "type": "string",
      "pattern": "^[a-z0-9]+(?:_[a-z0-9]+)*$",
      "description": "Main student problem or barrier, never the support/service offered"
    },
    "secondary_concerns": {
      "type": "array",
      "description": "Other explicit problems, excluding consequences of the primary concern",
      "items": {
        "type": "string",
        "pattern": "^[a-z0-9]+(?:_[a-z0-9]+)*$"
      }
    },
    "attendance_issue": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "reported_absences": {
          "type": [
            "integer",
            "null"
          ],
          "minimum": 0
        },
        "cause": {
          "type": [
            "string",
            "null"
          ]
        },
        "evidence_status": {
          "type": "string",
          "enum": [
            "student_reported",
            "mentor_r

## Ask the model for a conclusion

In [4]:
SYSTEM_PROMPT = """You extract conservative, evidence-grounded conclusions from synthetic mentor-student sessions.
Use only the supplied conversation. Never infer eligibility, diagnosis, household income, motives, delivery, or improvement.
A proposed/requested service is not a delivered service. If the transcript has no post-intervention observation,
intervention_helpful and relevant outcomes must be unknown. A session may still be helpful if the student explicitly
describes value or a concrete agreed next step. Use snake_case for concern, intervention, and next-step labels.
The primary_concern is the student's problem (for example transport_cost), not the proposed support.
Do not list attendance as a secondary concern when it is only a consequence of the primary barrier.
Use assessment_requested when the mentor agrees to request or records agreement to an assessment; use inquiry_planned
only when someone merely plans to ask or check. The next_step is the earliest agreed action, not a later follow-up.
Set reported_absences to null unless a total count is explicitly stated.
Evidence must contain only short, exact spoken words
copied from an utterance; omit speaker labels such as Mentor: or Student: and omit Markdown formatting."""

USER_PROMPT = f"""Read the conversation below and return its intervention-session conclusion.
Follow this JSON schema exactly:
{json.dumps(CONCLUSION_SCHEMA)}

CONVERSATION:
{conversation}"""

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    "stream": False,
    "think": False,
    "format": CONCLUSION_SCHEMA,
    "options": {"temperature": 0},
}

response = ollama_request("/api/chat", payload)
conclusion = json.loads(response["message"]["content"])

missing_keys = set(CONCLUSION_SCHEMA["required"]) - conclusion.keys()
if missing_keys:
    raise ValueError(f"Model output is missing required keys: {sorted(missing_keys)}")

# Deterministic evidence rule: a discussion or request cannot establish an intervention outcome.
if not conclusion["intervention"]["service_delivered"]:
    conclusion["outcome"]["primary_barrier_resolved"] = "unknown"
    conclusion["outcome"]["attendance_improved"] = "unknown"
    conclusion["helpfulness"]["intervention_helpful"] = "unknown"
    rationale = conclusion["helpfulness"]["rationale"].strip()
    if rationale:
        rationale = rationale[0].upper() + rationale[1:]
        if rationale[-1] not in ".!?":
            rationale += "."
    conclusion["helpfulness"]["rationale"] = (
        rationale + " No delivered service or post-intervention outcome is documented."
    )

print(json.dumps(conclusion, indent=2, ensure_ascii=False))

{
  "primary_concern": "transport_cost",
  "secondary_concerns": [],
  "attendance_issue": {
    "reported_absences": null,
    "cause": "transport_cost",
    "evidence_status": "student_reported"
  },
  "intervention": {
    "type": "transport_assessment",
    "status": "assessment_requested",
    "accepted_by_student": true,
    "service_delivered": false
  },
  "outcome": {
    "primary_barrier_resolved": "unknown",
    "attendance_improved": "unknown"
  },
  "helpfulness": {
    "session_helpful": "yes",
    "intervention_helpful": "unknown",
    "rationale": "Student explicitly describes value and agrees to assessment request. No delivered service or post-intervention outcome is documented."
  },
  "next_step": "request_transport_assessment",
  "evidence": [
    "The bus fare has become difficult to cover on some mornings.",
    "Last Thursday I waited at the stop, then went home because I had no fare.",
    "Would you like me to request an assessment of available support, without

## Quick evidence checks

These checks catch two especially important failure modes: invented quotations and claiming a demonstrated intervention effect before a service was delivered.

In [5]:
plain_conversation = conversation.replace("**Mentor:**", "Mentor:").replace("**Student:**", "Student:")
for quote in conclusion["evidence"]:
    assert quote in plain_conversation, f"Evidence is not a verbatim transcript excerpt: {quote!r}"

if not conclusion["intervention"]["service_delivered"]:
    assert conclusion["helpfulness"]["intervention_helpful"] == "unknown", (
        "Intervention helpfulness cannot be established before delivery."
    )

print("Checks passed: evidence is verbatim and helpfulness is conservatively reported.")

Checks passed: evidence is verbatim and helpfulness is conservatively reported.


## Try another conversation

Point `DIALOGUE_PATH` at any `.md` file below `src/intervention/student_mentor_dialogues/concerns`, then rerun from the load-transcript cell. Once the output shape looks right across several cases, the request, prompt, and checks can be moved into a reusable class.